In [6]:
from bs4 import BeautifulSoup
import requests
import json
import os


CURR_DIR = os.curdir
Links = ["https://www.ignou.ac.in/contactUs?nav=9", # Contact us
         ]


def scrape_contact_us(url , to_scrape):
    r = requests.get(url)
    soup = BeautifulSoup(r.text , "html.parser")

    article = []
    scraped_data = {}
    for para in soup.find_all(["p"]):
        article.append(para.text)
    scraped_data["Address"] = article[0].strip("\n").strip()
    scraped_data["Office Timings"] = article[1].strip("\n").strip().strip("Office Timing: ")

    head_office_data = {}
    IGNOU_head_office_number = soup.find_all("tbody")[0]
    for row in IGNOU_head_office_number.find_all("tr"):
        try:
            i , key , value = row.find_all("td")
            head_office_data[key.text.strip()] = value.text.strip()
        except:
            continue

    Regional_centres_official = IGNOU_head_office_number = soup.find_all("tbody")[1]
    centres_official_data = {}
    for row in Regional_centres_official.find_all("tr"):
            try:
                sno1 , region1 , num1 , sno2 , region2 , num2 = row.find_all("td")
                centres_official_data[region1.text.strip()] = num1.text.strip()
                centres_official_data[region2.text.strip()] = num2.text.strip()
            except:
                continue

    complete_data = {"Address" : scraped_data ,
                     "IGNOU Head Office Contact Number" : head_office_data ,
                     "Regional Centres Officials Contact Number" : centres_official_data}
    return complete_data


def save(data):
    with open(os.path.join(CURR_DIR,"Data","ContactUs.json"),"w") as F:
        json.dump(data,F,indent=4)

In [6]:
data = scrape_contact_us(Links[0] , 1)
print(data)

{'Address': {'Address': 'Indira Gandhi National Open University Maidan Garhi, New Delhi, India  Pin Code: 110068', 'Office Timings': '9:30 AM to 6:00 PM (Monday to Friday)'}, 'IGNOU Head Office Contact Number': {'IGNOU Telephone Exchange': '29571000', 'Student Service Centre (SSC)': 'Student Enquiry Numbers:29572513, 29572514', 'Material Production & Distribution Division (MPDD)': 'For Study Materials: 29572008, 29572012, 29572013.\r\n                                                                            Additional Numbers: 29534521, 29572002', 'Certificate Programmes': '29572208', 'Diploma Programmes': '29572208', 'Bachelors Programmes (BA/B.Sc./B.Com)': '29572211', 'Masters Programmes & Bachelors (B.Ed, BTS, BHM, BCA, BSW, BLIS, BSCHOT, BSCN,\r\n                                                                            BTME)': '29572212', 'Issuance of Degree Certificate': '29572224, 29535438', 'Issue of Transcript and verification of Degree Certificate': '29571524', 'Assignment

In [10]:
save(data)

In [15]:
from urllib.parse import urljoin
import requests
import os


def download_pdfs(page_url , num_papers , session , save_folder="Data/PDFs"):
    """
    Downloads all PDFs linked on the given webpage.
    """
    os.makedirs(save_folder, exist_ok=True)
    response = requests.get(page_url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    downloaded = []
    for link in soup.find_all("a", href=True)[:num_papers]:
        href = link["href"]
        if ".pdf" not in href.lower():
            continue
        pdf_url = urljoin(page_url, href)
        filename = os.path.basename(pdf_url.split("?")[0])
        if filename == "":
            filename = f"file_{len(downloaded)+1}.pdf"
        filepath = os.path.join(save_folder, filename)
        print(f"Downloading {filename}...")
        try:
            pdf_response = requests.get(pdf_url, stream=True)
            pdf_response.raise_for_status()
            with open(filepath[:-4]+"  "+session+".pdf", "wb") as f:
                for chunk in pdf_response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            downloaded.append(filepath[:-4]+"-"+session+".pdf")
        except Exception as e:
            print(f"Failed to download {pdf_url}")
            print(e)
    print(f"\nDownloaded {len(downloaded)} PDFs.")
    return downloaded

In [16]:
pyqs = {"June 2025" : "https://www.ignou.ac.in/studentService/download/questionPapers/June2025",
        "December 2025" : "https://www.ignou.ac.in/studentService/download/questionPapers/Dec2025",
        "December 2024" : "https://www.ignou.ac.in/studentService/download/questionPapers/Dec2024"}

In [18]:
for session in pyqs:
    download_pdfs(pyqs[session] , 400 , session)


Downloaded 368 PDFs.

Downloaded 368 PDFs.

Downloaded 368 PDFs.


In [ ]:
def get_pdf_links(link):
    """Lists all the papers/pyqs for the given time
argument is the link provided in front of the time 
"June 2025" : "https://www.ignou.ac.in/studentService/download/questionPapers/June2025",
"December 2025" : "https://www.ignou.ac.in/studentService/download/questionPapers/Dec2025",
"December 2024" : "https://www.ignou.ac.in/studentService/download/questionPapers/Dec2024"]
    """
    

In [ ]:
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup

def get_pdf_links(link):
    """
    Returns a dictionary:
    {
        "filename.pdf": "https://....pdf",
        ...
    }
    """

    response = requests.get(link)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    pdf_links = {}
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if not href.lower().endswith(".pdf"):
            continue
        pdf_url = urljoin(link, href)
        
        # Get filename
        filename = pdf_url.split("/")[-1].split("?")[0]

        # Avoid duplicate keys
        if filename in pdf_links:
            i = 1
            name = filename.rsplit(".", 1)[0]
            ext = filename.rsplit(".", 1)[1]
            while f"{name}_{i}.{ext}" in pdf_links:
                i += 1
            filename = f"{name}_{i}.{ext}"

        pdf_links[filename] = pdf_url

    return pdf_links

In [126]:
links = get_pdf_links(
    "https://www.ignou.ac.in/studentService/download/questionPapers/Dec2024"
)

print(f"Found {len(links)} PDFs")

for name, url in list(links.items())[:5]:
    print(name)
    print(url)

Found 2605 PDFs
ACC-1.pdf
https://www.ignou.ac.in/viewFile/ldd/downloads/ACC-1.pdf
ACS-01.pdf
https://www.ignou.ac.in/viewFile/ldd/downloads/ACS-01.pdf
ACS-01_ORIYA.pdf
https://www.ignou.ac.in/viewFile/ldd/downloads/ACS-01_ORIYA.pdf
AEC-01.pdf
https://www.ignou.ac.in/viewFile/ldd/downloads/AEC-01.pdf
AED-01.pdf
https://www.ignou.ac.in/viewFile/ldd/downloads/AED-01.pdf


In [127]:
with open(os.path.join(CURR_DIR,"Data","December2024.json"),"w") as F:
    json.dump(links,F,indent=4)

In [128]:
with open(os.path.join(CURR_DIR,"Data","December2024.json"),"r") as F:
    t = json.load(F)